In [17]:
import gymnasium as gym
import numpy as np
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim

In [18]:
# ======================
# Hyperparameters
# ======================
ENV_NAME = "LunarLander-v3"
GAMMA = 0.99
LR = 1e-3
BATCH_SIZE = 64
BUFFER_SIZE = 100_000
MIN_REPLAY_SIZE = 1_000
TAU = 0.01                   # soft update rate
EPS_START = 1.0
EPS_END = 0.01
EPS_DECAY = 0.995
NUM_EPISODES = 200
MAX_STEPS = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
# ======================
# Q-Network
# ======================
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.net(x)

In [20]:
# ======================
# Replay Buffer
# ======================
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            torch.tensor(states, dtype=torch.float32, device=device),
            torch.tensor(actions, dtype=torch.int64, device=device),
            torch.tensor(rewards, dtype=torch.float32, device=device),
            torch.tensor(next_states, dtype=torch.float32, device=device),
            torch.tensor(dones, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)

In [21]:
# ======================
# Epsilon-Greedy Policy
# ======================
def select_action(state, q_net, epsilon, action_dim):
    if random.random() < epsilon:
        return random.randrange(action_dim)
    else:
        state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            q_values = q_net(state)
        return torch.argmax(q_values).item()


In [22]:
# ======================
# Training Step
# ======================
def train_step(q_net, target_net, optimizer, replay_buffer):
    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    # Current Q(s,a)
    q_values = q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    # Target Q
    with torch.no_grad():
        max_next_q = target_net(next_states).max(1)[0]
        target_q = rewards + GAMMA * (1 - dones) * max_next_q

    loss = nn.MSELoss()(q_values, target_q)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [23]:
# ======================
# Soft Update
# ======================
def soft_update(target_net, q_net, tau):
    for target_param, param in zip(target_net.parameters(), q_net.parameters()):
        target_param.data.copy_(
            tau * param.data + (1 - tau) * target_param.data
        )

In [24]:
# ======================
# Main Training Loop
# ======================
env = gym.make(ENV_NAME)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

q_net = QNetwork(state_dim, action_dim).to(device)
target_net = QNetwork(state_dim, action_dim).to(device)
target_net.load_state_dict(q_net.state_dict())

optimizer = optim.Adam(q_net.parameters(), lr=LR)
replay_buffer = ReplayBuffer(BUFFER_SIZE)

epsilon = EPS_START
scores = []

for episode in range(NUM_EPISODES):
    state, _ = env.reset()
    episode_reward = 0

    for step in range(MAX_STEPS):
        action = select_action(state, q_net, epsilon, action_dim)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        replay_buffer.add(state, action, reward, next_state, done)
        state = next_state
        episode_reward += reward

        if len(replay_buffer) >= MIN_REPLAY_SIZE:
            train_step(q_net, target_net, optimizer, replay_buffer)
            soft_update(target_net, q_net, TAU)

        if done:
            break

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    scores.append(episode_reward)

    print(f"Episode {episode:4d} | Reward: {episode_reward:7.2f} | Epsilon: {epsilon:.3f}")

env.close()

Episode    0 | Reward:  -58.22 | Epsilon: 0.995
Episode    1 | Reward: -365.98 | Epsilon: 0.990
Episode    2 | Reward: -335.13 | Epsilon: 0.985
Episode    3 | Reward: -172.17 | Epsilon: 0.980
Episode    4 | Reward: -244.25 | Epsilon: 0.975
Episode    5 | Reward: -425.94 | Epsilon: 0.970
Episode    6 | Reward:  -42.75 | Epsilon: 0.966
Episode    7 | Reward: -284.03 | Epsilon: 0.961
Episode    8 | Reward: -438.41 | Epsilon: 0.956
Episode    9 | Reward: -120.16 | Epsilon: 0.951
Episode   10 | Reward:  -40.61 | Epsilon: 0.946
Episode   11 | Reward:  -45.53 | Epsilon: 0.942
Episode   12 | Reward: -197.25 | Epsilon: 0.937
Episode   13 | Reward: -342.74 | Epsilon: 0.932
Episode   14 | Reward: -206.79 | Epsilon: 0.928
Episode   15 | Reward: -169.66 | Epsilon: 0.923
Episode   16 | Reward: -135.85 | Epsilon: 0.918
Episode   17 | Reward:  -88.18 | Epsilon: 0.914
Episode   18 | Reward:  -70.42 | Epsilon: 0.909
Episode   19 | Reward:  -85.44 | Epsilon: 0.905
Episode   20 | Reward: -221.11 | Epsilon